
# Laboratorium — Sieci neuronowe i propagacja wsteczna
## Implementacja sieci neuronowej od zera: XOR + opcjonalnie MNIST

### Cele laboratorium
Po wykonaniu laboratorium powinieneś/powinnaś umieć:

- zaimplementować **forward pass**,
- policzyć **funkcję kosztu**,
- wyprowadzić i zaimplementować **backpropagation**,
- zaktualizować wagi metodą **gradient descent**,
- wytrenować prostą sieć neuronową dla problemu **XOR**,
- opcjonalnie uruchomić tę samą architekturę dla uproszczonego zbioru **MNIST**.

### Założenia
- używamy tylko bibliotek ogólnego przeznaczenia,
- **nie używamy bibliotek ML** typu scikit-learn / tensorflow / pytorch,
- implementujemy sieć od zera z użyciem `numpy`.


In [ ]:

from __future__ import annotations

from dataclasses import dataclass
import csv

import numpy as np
import matplotlib.pyplot as plt



## 1. Dane — XOR

Problem XOR:

- `(0, 0) -> 0`
- `(0, 1) -> 1`
- `(1, 0) -> 1`
- `(1, 1) -> 0`

To klasyczny przykład problemu **nieliniowo separowalnego**.


In [ ]:

X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

y = np.array([
    [0.0],
    [1.0],
    [1.0],
    [0.0],
])

print("X shape:", X.shape)
print("y shape:", y.shape)
print()
print("X:")
print(X)
print()
print("y:")
print(y)



## 2. Funkcje aktywacji

Na potrzeby laboratorium użyjemy funkcji **sigmoid**.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

oraz jej pochodnej:

$$\sigma'(z) = \sigma(z)(1-\sigma(z))$$

W implementacji wygodnie liczyć pochodną na podstawie już policzonej aktywacji.


In [ ]:

def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_derivative_from_activation(a: np.ndarray) -> np.ndarray:
    return a * (1.0 - a)



## 3. Architektura sieci

Zaimplementujemy prostą sieć:

- **2 neurony wejściowe**
- **2 neurony w warstwie ukrytej**
- **1 neuron wyjściowy**

Schemat:


$$2 \rightarrow 2 \rightarrow 1$$

W notacji:
- `W1`, `b1` — warstwa ukryta
- `W2`, `b2` — warstwa wyjściowa


In [ ]:

@dataclass
class Parameters:
    W1: np.ndarray
    b1: np.ndarray
    W2: np.ndarray
    b2: np.ndarray


@dataclass
class Cache:
    Z1: np.ndarray
    A1: np.ndarray
    Z2: np.ndarray
    A2: np.ndarray


@dataclass
class Gradients:
    dW1: np.ndarray
    db1: np.ndarray
    dW2: np.ndarray
    db2: np.ndarray



## 4. Inicjalizacja parametrów

Wagi inicjalizujemy losowo małymi wartościami. Biasy zaczynają od zera.


In [ ]:

def init_parameters(input_size: int, hidden_size: int, output_size: int, seed: int = 42) -> Parameters:
    rng = np.random.default_rng(seed)

    W1 = rng.normal(loc=0.0, scale=0.5, size=(input_size, hidden_size))
    b1 = np.zeros((1, hidden_size))
    W2 = rng.normal(loc=0.0, scale=0.5, size=(hidden_size, output_size))
    b2 = np.zeros((1, output_size))

    return Parameters(W1=W1, b1=b1, W2=W2, b2=b2)


params = init_parameters(input_size=2, hidden_size=2, output_size=1)
print("W1 shape:", params.W1.shape)
print("b1 shape:", params.b1.shape)
print("W2 shape:", params.W2.shape)
print("b2 shape:", params.b2.shape)



## 5. Forward pass

Dla całej macierzy wejść liczymy:

$$Z^{[1]} = XW^{[1]} + b^{[1]}$$

$$A^{[1]} = \sigma(Z^{[1]})$$

$$Z^{[2]} = A^{[1]}W^{[2]} + b^{[2]}$$

$$A^{[2]} = \sigma(Z^{[2]})$$

`A2` to przewidywanie modelu.


In [ ]:

def forward_pass(X: np.ndarray, params: Parameters) -> Cache:
    Z1 = X @ params.W1 + params.b1
    A1 = sigmoid(Z1)

    Z2 = A1 @ params.W2 + params.b2
    A2 = sigmoid(Z2)

    return Cache(Z1=Z1, A1=A1, Z2=Z2, A2=A2)


cache = forward_pass(X, params)
print("A2:")
print(cache.A2)



## 6. Funkcja kosztu

Używamy **MSE**:

$$
L = \frac{1}{n}\sum (\hat{y} - y)^2
$$


In [ ]:

def mse_loss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_pred - y_true) ** 2))


initial_loss = mse_loss(y, cache.A2)
print("Initial loss:", initial_loss)



## 7. Backpropagation

Dla naszej sieci:

### Warstwa wyjściowa
$$dA2 = \frac{2}{n}(A2 - y)$$

$$dZ2 = dA2 \odot \sigma'(A2)$$

$$dW2 = A1^T dZ2$$

$$db2 = \sum dZ2$$

### Warstwa ukryta
$$dA1 = dZ2 W2^T$$

$$dZ1 = dA1 \odot \sigma'(A1)$$

$$dW1 = X^T dZ1$$

$$db1 = \sum dZ1$$


In [ ]:

def backward_pass(X: np.ndarray, y: np.ndarray, params: Parameters, cache: Cache) -> Gradients:
    ## TODO Implementacja algorytmu wstecznej propagacji
    pass


grads = backward_pass(X, y, params, cache)
print("dW1 shape:", grads.dW1.shape)
print("db1 shape:", grads.db1.shape)
print("dW2 shape:", grads.dW2.shape)
print("db2 shape:", grads.db2.shape)



## 8. Aktualizacja parametrów

Stosujemy prosty **gradient descent**:

$$W := W - \eta dW$$

$$b := b - \eta db$$


In [ ]:

def update_parameters(params: Parameters, grads: Gradients, lr: float) -> Parameters:
    return Parameters(
        W1=params.W1 - lr * grads.dW1,
        b1=params.b1 - lr * grads.db1,
        W2=params.W2 - lr * grads.dW2,
        b2=params.b2 - lr * grads.db2,
    )



## 9. Pętla treningowa

W każdej epoce wykonujemy:
1. forward pass
2. obliczenie loss
3. backward pass
4. update


In [ ]:

def train_xor(
    X: np.ndarray,
    y: np.ndarray,
    hidden_size: int = 2,
    lr: float = 1.0,
    epochs: int = 10_000,
    seed: int = 42,
    verbose_every: int = 1000,
):
    params = init_parameters(
        input_size=X.shape[1],
        hidden_size=hidden_size,
        output_size=y.shape[1],
        seed=seed,
    )

    losses = []

    for epoch in range(1, epochs + 1):
        cache = forward_pass(X, params)
        loss = mse_loss(y, cache.A2)
        losses.append(loss)

        grads = backward_pass(X, y, params, cache)
        params = update_parameters(params, grads, lr=lr)

        if verbose_every and epoch % verbose_every == 0:
            print(f"Epoch {epoch:5d} | loss = {loss:.6f}")

    return params, losses


In [ ]:

params, losses = train_xor(
    X,
    y,
    hidden_size=2,
    lr=1.0,
    epochs=10_000,
    seed=42,
    verbose_every=1000,
)



## 10. Wykres funkcji kosztu


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.title("Uczenie sieci dla XOR")
plt.grid(True)
plt.show()



## 11. Predykcja i ewaluacja


In [ ]:

def predict_proba(X: np.ndarray, params: Parameters) -> np.ndarray:
    return forward_pass(X, params).A2


def predict_binary(X: np.ndarray, params: Parameters, threshold: float = 0.5) -> np.ndarray:
    probs = predict_proba(X, params)
    return (probs >= threshold).astype(int)


y_prob = predict_proba(X, params)
y_pred = predict_binary(X, params)

print("Prawdopodobieństwa:")
print(np.round(y_prob, 4))
print()
print("Predykcje:")
print(y_pred)
print()
print("Prawdziwe etykiety:")
print(y.astype(int))

accuracy = np.mean(y_pred == y)
print()
print(f"Accuracy: {accuracy:.2%}")



## 12. Wizualizacja granicy decyzyjnej


In [ ]:

def plot_decision_boundary(params: Parameters, X: np.ndarray, y: np.ndarray, steps: int = 200) -> None:
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5

    xs = np.linspace(x_min, x_max, steps)
    ys = np.linspace(y_min, y_max, steps)
    xx, yy = np.meshgrid(xs, ys)

    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = predict_proba(grid, params).reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, probs, levels=30, alpha=0.7)
    plt.colorbar(label="P(y=1)")
    plt.contour(xx, yy, probs, levels=[0.5], linewidths=2)

    for i in range(len(X)):
        marker = "o" if y[i, 0] == 0 else "s"
        plt.scatter(X[i, 0], X[i, 1], marker=marker, s=120)

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title("Granica decyzyjna dla XOR")
    plt.grid(True)
    plt.show()


plot_decision_boundary(params, X, y)
